# 🧪 معمل التجارب: بناء سلاسل RAG بـ FAISS و LCEL (Lecture 33)
أهلاً بك في معملك التطبيقي! 🚀

**فكرة المعمل ببساطة:**
في الدرس السابق بنينا وحفظنا قاعدة متجهات `FAISS`. النهاردة هنحول الفهرس ده لـ **شات بوت RAG ذكي** باستخدام مشغل الربط `|` في **LCEL** وثلاث سلاسل مختلفة:
1. **سلسلة بسيطة (Simple RAG):** سؤال وجواب مباشر.
2. **سلسلة متدفقة (Streaming RAG):** الكلام بيطلع كلمة بكلمة زي ChatGPT.
3. **سلسلة محادثة (Conversational RAG):** بتفتكر الكلام السابق وبتفهم الضمائر.

> 💡 **طريقة التدريب في المعمل:**
> اقرأ الشرح المبسط ⬅️ شوف الكود المرجعي الجاهز ⬅️ واكتب الكود وجربه بإيدك في الخلية الفارغة المخصصة لك تحت كل خطوة!

### 1. استيراد المكتبات وتحميل المفتاح
**الفكرة ببساطة:**
بنجيب مفتاح Groq السري من ملف `.env`، وبنستورد مكتبات لانج تشين الأساسية اللي هنركب بيها السلسلة زي المكعبات.

In [ ]:
# 1. import standard libraries and load environment variables
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# resolve centralized faiss database path
data_dir = Path("../../data") if Path("../../data").exists() else Path("./data")
faiss_db_dir = str(data_dir / "databases" / "faiss_index")

# import langchain components
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage

print("✅ Libraries imported successfully!")
print(f"📁 Database Path: {faiss_db_dir}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب استيراد المكتبات وتحديد المسار بنفسك هنا):



### 2. تحميل فهرس FAISS وتجهيز المسترجع (Retriever)
**الفكرة ببساطة:**
بنفتح الفهرس اللي حفظناه على الهارد في الدرس اللي فات (`load_local`)، وبنحوله لـ `Retriever` يجيب لنا أقرب 3 قطع (`k=3`) لأي سؤال نسأله.

In [ ]:
# initialize embedding model to read vectors
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# load stored faiss index from disk
vector_store = FAISS.load_local(
    faiss_db_dir,
    embeddings,
    allow_dangerous_deserialization=True
)

# convert to retriever fetching top 3 chunks
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print(f"✅ FAISS Index Loaded! Total vectors: {vector_store.index.ntotal}")
print(f"🔍 Retriever is ready with k={retriever.search_kwargs['k']}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تحميل الفهرس وتجهيز الـ retriever بنفسك هنا):



### 3. تشغيل نموذج Groq فائق السرعة
**الفكرة ببساطة:**
بنشغل موديل الذكاء الاصطناعي من منصة Groq مع `temperature=0.2` (حرارة منخفضة) عشان يجاوب بدقة وبدون أي تأليف.

In [ ]:
# initialize fast Groq LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)
print("🚀 Groq LLM initialized and ready!")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تشغيل الـ LLM بنفسك هنا):



### 4. دالة ترتيب وتنسيق المستندات (`format_docs`)
**الفكرة ببساطة:**
الـ Retriever بيرجع لنا لستة من المستندات. الدالة دي وظيفتها ترتبهم في نص واحد نظيف ومكتوب فيه اسم المصدر والمجال عشان الموديل يفهمه بسهولة.

In [ ]:
# format retrieved documents into structured context text
def format_docs(docs):
    chunks_text = []
    for i, doc in enumerate(docs, start=1):
        src = doc.metadata.get("source", "Unknown")
        top = doc.metadata.get("topic", "General")
        chunks_text.append(f"[Document {i} | Source: {src} | Topic: {top}]:\n{doc.page_content}")
    return "\n\n".join(chunks_text)

# test formatting with a quick query
test_docs = retriever.invoke("What is machine learning?")
print(format_docs(test_docs)[:200], "...")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب كتابة دالة format_docs واختبارها هنا):



### 5. السلسلة الأولى: مسار RAG البسيط (Simple RAG Chain)
**الفكرة ببساطة:**
بنركب السلسلة بمشغل `|`:  
`(بحث في FAISS + تمرير السؤال) ⬅️ قالب البرومبت ⬅️ الموديل LLM ⬅️ مخرج نصي صافي`

In [ ]:
# 1. prompt template
simple_template = """You are a helpful assistant. Use ONLY the following context to answer the question.
If you don't know the answer, say "I don't know based on the provided documents."

Context:
{context}

Question: {question}
Answer:"""
simple_prompt = ChatPromptTemplate.from_template(simple_template)

# 2. assemble LCEL chain
simple_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
    | StrOutputParser()
)

# 3. invoke query
query = "What is the difference between AI and machine learning?"
response = simple_rag_chain.invoke(query)
print(f"❓ Question: {query}\n")
print(f"💬 Answer:\n{response}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب بناء وتشغيل simple_rag_chain بنفسك هنا):



### 6. السلسلة الثانية: البث اللحظي للكلمات (Streaming RAG Chain)
**الفكرة ببساطة:**
بدل ما تستنى الموديل يخلص الإجابة كاملة، بنستخدم `.stream()` عشان الكلام يظهر لك حرف بحرف وكلمة بكلمة في نفس اللحظة!

In [ ]:
# test streaming response chunk by chunk
stream_query = "What is deep learning and how does it relate to neural networks?"
print(f"❓ Question: {stream_query}\n")
print("⚡ Live Streaming Output: ")

for chunk in simple_rag_chain.stream(stream_query):
    print(chunk, end="", flush=True)

print("\n\n✅ [Streaming Finished!]")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تشغيل الـ Streaming بنفسك هنا):



### 7. السلسلة الثالثة: شات تفاعلي يحفظ الذاكرة (Conversational RAG Chain)
**الفكرة ببساطة:**
بنحط خانة مخصصة في البرومبت اسمها `MessagesPlaceholder("chat_history")` عشان الموديل يقرأ الشات القديم قبل ما يجاوب على السؤال الجديد.

In [ ]:
# conversational prompt with chat history placeholder
conv_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Answer using the provided context and history.\n\nContext:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}")
])

# conversational LCEL chain
conversational_rag_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["question"]))
    )
    | conv_prompt
    | llm
    | StrOutputParser()
)
print("🧠 Conversational RAG chain is ready!")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب بناء conversational_rag_chain بنفسك هنا):



### 8. اختبار الشات: فهم الضمائر وحفظ المحادثة
**الفكرة ببساطة:**
- نسأل أول سؤال: *"What is machine learning?"* ونحفظ الرد في `chat_history`.
- نسأل سؤال تالي ناقص بالضمير: *"How is it different from traditional programming?"*.
- الموديل هيفهم فوراً إن كلمة `it` تعود على `machine learning` من تاريخ الشات!

In [ ]:
# initialize chat memory list
chat_history = []

# turn 1: initial question
q1 = "What is machine learning?"
a1 = conversational_rag_chain.invoke({"question": q1, "chat_history": chat_history})
print(f"👤 Human (Turn 1): {q1}\n")
print(f"🤖 Assistant (Turn 1):\n{a1}\n")

# update memory
chat_history.extend([HumanMessage(content=q1), AIMessage(content=a1)])

# turn 2: follow-up with pronoun 'it'
q2 = "How is it different from traditional programming?"
a2 = conversational_rag_chain.invoke({"question": q2, "chat_history": chat_history})
print("=" * 60)
print(f"👤 Human (Turn 2): {q2}\n")
print(f"🤖 Assistant (Turn 2):\n{a2}\n")

# update memory again
chat_history.extend([HumanMessage(content=q2), AIMessage(content=a2)])
print(f"Memory now holds {len(chat_history)} messages!")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب جولات المحادثة بنفسك هنا وتحديث الـ chat_history):



### 9. اختبار أسئلة متعددة في حلقة تكرارية (Batch Testing)
**الفكرة ببساطة:**
بنمرر قائمة من الأسئلة المتنوعة على السلسلة في `for loop` للتأكد من إن النظام شغال بثبات وسرعة مع مختلف الأسئلة.

In [ ]:
# test multiple questions in a loop
test_questions = [
    "What is natural language processing and what are its core tasks?",
    "What architectures are used in deep learning for visual and textual data?"
]

for idx, q in enumerate(test_questions, start=1):
    print(f"❓ Question [{idx}]: {q}")
    ans = simple_rag_chain.invoke(q)
    print(f"💬 Answer:\n{ans}\n" + "-" * 50)

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب حلقة اختبار بأسئلتك الخاصة هنا):



## 🎯 ملخص المعمل السريع:
1. **FAISS:** أسرع محرك بحث دلالي في الذاكرة الحية.
2. **Simple RAG:** سؤال وجواب مباشر.
3. **Streaming:** بيبث الرد لحظة بلحظة عبر `.stream()`.
4. **Conversational RAG:** بيفهم الضمائر والسياق عبر `chat_history`.

عاش جداً يا بطل! 👏 دلوقتي تقدر ترجع وتجرب كتابة الكود بنفسك في الخلايا المخصصة لك فوق!